In [ ]:
import numpy as np
from tqdm import tqdm
from itertools import combinations

In [3]:
# Input parameters
P = int(input("Enter number of parties (P): "))  # Number of parties
I = int(input("Enter number of interventions (I): "))  # Number of interventions per party
O = int(input("Enter number of outcomes (O): "))  # Number of outcomes per party

In [4]:
# Helper function to generate output combinations (O^P)
def OPlist():
    OP = []
    for i in range(O**P):
        changedBase = 0
        temp = i
        count = 0
        while temp > 0:
            rem = temp % O
            changedBase += rem * (10**count)
            count += 1
            temp = temp // O
        OP += [(P - len(str(changedBase))) * '0' + str(changedBase)]
    return OP

# Helper function to generate input combinations (I^P)
def IPlist():
    IP = []
    for i in range(I**P):
        changedBase = 0
        temp = i
        count = 0
        while temp > 0:
            rem = temp % I
            changedBase += rem * (10**count)
            count += 1
            temp = temp // I
        IP += [(P - len(str(changedBase))) * '0' + str(changedBase)]
    return IP

In [5]:
# Function to change number base from base frombase to another base. m is padding
def ChangeBase2(num, frombase, tobase, m):
    changedBase = 0
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += rem * (frombase**count)
        count += 1
        temp = temp // tobase
    return (m - len(str(changedBase))) * '0' + str(changedBase)

# Function to change number base from base 10 to another base
def ChangeBase(num, tobase, m):
    changedBase = 0
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += rem * (10**count)
        count += 1
        temp = temp // tobase
    return (m - len(str(changedBase))) * '0' + str(changedBase)

# Function to change number base (as list)
def ChangeBaseList(num, tobase, m):
    changedBase = []
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += [rem]
        count += 1
        temp = temp // tobase
    temp=[]
    for i in range(m - len(changedBase)):
        temp+=[0]
    changedBase += temp
    return changedBase

# Initialize empty table
def Tables(OP, IP):
    TD = [[0 for _ in range(len(IP))] for _ in range(len(OP))]
    return TD

#Function to print table
def PrintTable(TD, OP, IP):
    print("OP\\IP", end='\t')
    for ip in IP:
        print(ip, end='\t')
    print()
    for i, op in enumerate(OP):
        print(op, end='\t')
        for j in range(len(IP)):
            print(TD[i][j], end='\t')
        print()

In [6]:
# Generic PNS checker
def check_possibilistic_no_signaling(Table, P, I, O):
    for i in range(P): #fixes (a,x) or (b,y) etc
        #print("Party:",i)
        for j in range(I): #fixes x or y etc
            for k in range(O): # fixes a or b etc
                MarginalList=[]
                for l in range(I): #fixes y or x resp. 
                    if i == 0: # if first party, then a
                        column= int(ChangeBase2(int(str(j)+str(l)),I,10,2))
                    else: # if not first party, then b
                        column= int(ChangeBase2(int(str(l)+str(j)),I,10,2))

                    Rows=[]
                    for temp in range(O): # takes all b or a resp.
                        if i == 0:
                            index= int(ChangeBase2(int(str(k)+str(temp)),O,10,2))
                            Rows+=[index]
                        else: # if not first party, then b
                            index= int(ChangeBase2(int(str(temp)+str(k)),O,10,2))
                            Rows+=[index]
                   #print("Rows:", Rows, "Column:", column)
                    marginal=0
                    for row in Rows:
                        marginal = (marginal or Table[row][column])
                        #marginal = (marginal or Table[column][row])
                    MarginalList+=[marginal]

                #print("MarginalList:", MarginalList)
                if (0 in MarginalList) and (1 in MarginalList):
                    return False          
    return True
                    


# FD Tables

In [7]:
# Generate OP and IP lists
OP = OPlist()
IP = IPlist()

# Generate all FD codes
N = (O**P)**(I**P)
AllFDCode = [ChangeBase(i, O**P, I**P) for i in range(N)]

# Base FD mappings
fd = [[1 if i == j else 0 for j in range(O**P)] for i in range(O**P)]
print("All FD configurations for a single combination of intervention:")

# Validate all FD tables
AllFDTables = []
AllFDTablesCode=[]
for code in tqdm(AllFDCode):
    FD_table_transpose = [fd[int(c)] for c in code]
    FD_table = [[FD_table_transpose[j][i] for j in range(len(FD_table_transpose))]
                for i in range(len(FD_table_transpose[0]))]
    if check_possibilistic_no_signaling(FD_table, P, I, O):
       #PrintTable(FD_table, OP, IP)
       #do_continue=input("Continue checking? (yes/no): ").strip().lower()
       #if do_continue =='no':
           #break
        AllFDTables.append(FD_table)
        AllFDTablesCode.append(code)

# Results
print("Number of FD tables satisfying PNS:", len(AllFDTables))
print("Total number of FD tables:", len(AllFDCode))
print("All FD tables:")
for table in AllFDTables:
    #print(table)
    PrintTable(table, OP, IP)


All FD configurations for a single combination of intervention:


100%|██████████| 256/256 [00:00<00:00, 20547.33it/s]

Number of FD tables satisfying PNS: 16
Total number of FD tables: 256
All FD tables:
OP\IP	00	01	10	11	
00	1	1	1	1	
01	0	0	0	0	
10	0	0	0	0	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	1	1	0	0	
01	0	0	0	0	
10	0	0	1	1	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	1	0	1	0	
01	0	1	0	1	
10	0	0	0	0	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	0	0	1	0	
11	0	0	0	1	
OP\IP	00	01	10	11	
00	0	1	0	1	
01	1	0	1	0	
10	0	0	0	0	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	0	1	0	0	
01	1	0	0	0	
10	0	0	0	1	
11	0	0	1	0	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	1	1	
10	0	0	0	0	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	0	0	0	0	
11	0	0	1	1	
OP\IP	00	01	10	11	
00	0	0	1	1	
01	0	0	0	0	
10	1	1	0	0	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	1	1	1	
11	0	0	0	0	
OP\IP	00	01	10	11	
00	0	0	1	0	
01	0	0	0	1	
10	1	0	0	0	
11	0	1	0	0	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	0	1	0	
11	0	1	0	1	
OP\IP	00	01	10	11	
00	0	0	0	1	
01	0	0	1	0	
10	0	1	0	0	
11	1	0	0	0	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
1

In [9]:
print("All FD table codes:")
for code in AllFDTablesCode:
    PrintTable(AllFDTables[AllFDTablesCode.index(code)], OP, IP)
    print(code) 

All FD table codes:
OP\IP	00	01	10	11	
00	1	1	1	1	
01	0	0	0	0	
10	0	0	0	0	
11	0	0	0	0	
0000
OP\IP	00	01	10	11	
00	1	1	0	0	
01	0	0	0	0	
10	0	0	1	1	
11	0	0	0	0	
0022
OP\IP	00	01	10	11	
00	1	0	1	0	
01	0	1	0	1	
10	0	0	0	0	
11	0	0	0	0	
0101
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	0	0	1	0	
11	0	0	0	1	
0123
OP\IP	00	01	10	11	
00	0	1	0	1	
01	1	0	1	0	
10	0	0	0	0	
11	0	0	0	0	
1010
OP\IP	00	01	10	11	
00	0	1	0	0	
01	1	0	0	0	
10	0	0	0	1	
11	0	0	1	0	
1032
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	1	1	
10	0	0	0	0	
11	0	0	0	0	
1111
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	0	0	0	0	
11	0	0	1	1	
1133
OP\IP	00	01	10	11	
00	0	0	1	1	
01	0	0	0	0	
10	1	1	0	0	
11	0	0	0	0	
2200
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	1	1	1	
11	0	0	0	0	
2222
OP\IP	00	01	10	11	
00	0	0	1	0	
01	0	0	0	1	
10	1	0	0	0	
11	0	1	0	0	
2301
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	0	1	0	
11	0	1	0	1	
2323
OP\IP	00	01	10	11	
00	0	0	0	1	
01	0	0	1	0	
10	0	1	0	0	
11	1	0	0	0	
3210
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
1

# PNS Tables

In [ ]:
# Generate output combinations (O^P)
OP = OPlist()
# Generate input combinations (I^P)
IP = IPlist()
# Initialize empty PNS table
PNS = Tables(OP, IP)

# Generate all possible PNS configurations for a single combination of intervention
pns = []
num_outputs = O**P  # Total number of output configurations
for i in range(1,2**num_outputs):  # 2 options for each output (possible/not possible)
    changed_base = ChangeBase(i, tobase=2, m=num_outputs)
    pns.append([int(digit) for digit in changed_base])
print("All PNS configurations for a single combination of intervention:")
print(pns)

# Generate all possible PNS tables across all interventions
AllPNSCode = []
N = (2**num_outputs-1)**(I**P)  # Total number of PNS tables
n = 2**num_outputs-1
m = I**P

# Generate all codes (this is time-intensive; vectorization used later)
print("Generating all PNS tables...")
for i in tqdm(range(N)):
    changed_base = ChangeBaseList(i, n, m)
    AllPNSCode.append(changed_base)

# Validate PNS tables for Possibilistic No-Signaling
AllPNSTables = []
print("Validating PNS tables...")
for code in tqdm(AllPNSCode):
    # Convert the code into a matrix
    PNS_table_transpose = [pns[int(c)] for c in code]
    
    # Transpose the table to its correct structure
    PNS_table = [
        [PNS_table_transpose[j][i] for j in range(len(PNS_table_transpose))]
        for i in range(len(PNS_table_transpose[0]))
    ]
    
    # Check if the table satisfies Possibilistic No-Signaling
    if check_possibilistic_no_signaling(PNS_table, P, I, O):
        AllPNSTables.append(PNS_table)

# Results
print(f"Number of PNS tables satisfying PNS: {len(AllPNSTables)}")
print(f"Total number of PNS tables: {len(AllPNSCode)}")
for table in AllPNSTables:
    #print(table)
    PrintTable(table, OP, IP)

All PNS configurations for a single combination of intervention:
[[0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 1, 1], [0, 1, 0, 0], [0, 1, 0, 1], [0, 1, 1, 0], [0, 1, 1, 1], [1, 0, 0, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 0, 1, 1], [1, 1, 0, 0], [1, 1, 0, 1], [1, 1, 1, 0], [1, 1, 1, 1]]
Generating all PNS tables...


100%|██████████| 50625/50625 [00:00<00:00, 259725.67it/s]


Validating PNS tables...


100%|██████████| 50625/50625 [00:02<00:00, 20177.71it/s]


Number of PNS tables satisfying PNS: 2961
Total number of PNS tables: 50625
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	0	0	0	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	0	0	0	0	
11	0	0	1	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	0	0	0	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	0	1	0	
11	0	1	0	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	0	0	1	0	
11	0	0	0	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	1	0	1	0	
11	0	1	0	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	0	1	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	1	1	0	0	
10	0	0	1	0	
11	0	0	1	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	1	0	1	0	
11	0	1	1	1	
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	1	0	1	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	0	0	1	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	1	0	1	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	1	1	0	0	
10	0	0	1	0	
11	1	1	1	1	
OP\IP	00	01	10	11	
00	1	0	0	0	
01	1	1	0	0	
10	1	0	1	0

In [9]:
print(AllPNSTables)

[[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[0, 0, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 1, 1]], [[0, 0, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1]], [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], [[1, 0, 0, 0], [0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 1, 0], [1, 1, 1, 1]], [[1, 0, 0, 0], [1, 1, 0, 0], [0, 0, 1, 0], [0, 0, 1, 1]], [[0, 0, 0, 0], [1, 1, 0, 0], [1, 0, 1, 0], [0, 1, 1, 1]], [[0, 0, 0, 0], [1, 1, 0, 0], [1, 0, 1, 0], [1, 1, 1, 1]], [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 1, 1, 1]], [[1, 0, 0, 0], [0, 1, 0, 0], [1, 0, 1, 0], [1, 1, 1, 1]], [[1, 0, 0, 0], [1, 1, 0, 0], [0, 0, 1, 0], [1, 1, 1, 1]], [[1, 0, 0, 0], [1, 1, 0, 0], [1, 0, 1, 0], [0, 1, 1, 1]], [[1, 0, 0, 0], [1, 1, 0, 0], [1, 0, 1, 0], [1, 1, 1, 1]], [[0, 0, 0, 0], [0, 0, 0, 0], [0, 1, 0, 1], [1, 0, 1, 0]], [[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], [[0, 1, 0, 0]

# ORs of FD tables

In [10]:
import itertools

def powerset(iterable):
    """
    Generates all non-empty subsets (as tuples) of the given iterable.
    For example, powerset([a, b, c]) returns:
      (a,), (b,), (c,), (a, b), (a, c), (b, c), (a, b, c)
    """
    s = list(iterable)
    # Generate combinations for all possible lengths (from 1 to len(s))
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(1, len(s) + 1))

def or_tables(tables):
    """
    Given a list of 2-D tables (each table is a list of lists with 0s and 1s),
    computes the element-wise OR of these tables.
    Assumes that all tables have the same dimensions.
    """
    rows = len(tables[0])           # Number of rows in the first table
    cols = len(tables[0][0])        # Number of columns in the first table
    # Initialize the result table with zeros (same dimensions as the input tables)
    result = [[0] * cols for _ in range(rows)]
    # For every table in the list:
    for table in tables:
        # For every row index i:
        for i in range(rows):
            # For every column index j:
            for j in range(cols):
                # Compute the OR: if either the current result or table value is 1, result becomes 1.
                result[i][j] = result[i][j] or table[i][j]
    return result

tables=AllFDTables

AllUniqueORs=[]
# Loop through every non-empty subset of tables
for subset in tqdm(powerset(tables)):
    # Calculate the entry-wise OR for the current subset
    combined = or_tables(subset)
    
    if combined not in AllUniqueORs:
        AllUniqueORs.append(combined)

#To print all subsets and results uncomment the below:
'''
    print("Subset:")
    # Print each table in the subset
    for table in subset:
        for row in table:
            print(row)
        print("-----")
    
    print("OR result:")
    # Print the resulting table after performing entry-wise OR on the subset
    for row in combined:
        print(row)
    print("=======\n")
'''
# Results
print(f"Number of unique OR tables: {len(AllUniqueORs)}")
for i, table in enumerate(AllUniqueORs):
    print(f"Table {i + 1}")
    PrintTable(table, OP, IP)

65535it [00:06, 9889.33it/s] 


Number of unique OR tables: 1721
Table 1
OP\IP	00	01	10	11	
00	1	1	1	1	
01	0	0	0	0	
10	0	0	0	0	
11	0	0	0	0	
Table 2
OP\IP	00	01	10	11	
00	1	1	0	0	
01	0	0	0	0	
10	0	0	1	1	
11	0	0	0	0	
Table 3
OP\IP	00	01	10	11	
00	1	0	1	0	
01	0	1	0	1	
10	0	0	0	0	
11	0	0	0	0	
Table 4
OP\IP	00	01	10	11	
00	1	0	0	0	
01	0	1	0	0	
10	0	0	1	0	
11	0	0	0	1	
Table 5
OP\IP	00	01	10	11	
00	0	1	0	1	
01	1	0	1	0	
10	0	0	0	0	
11	0	0	0	0	
Table 6
OP\IP	00	01	10	11	
00	0	1	0	0	
01	1	0	0	0	
10	0	0	0	1	
11	0	0	1	0	
Table 7
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	1	1	
10	0	0	0	0	
11	0	0	0	0	
Table 8
OP\IP	00	01	10	11	
00	0	0	0	0	
01	1	1	0	0	
10	0	0	0	0	
11	0	0	1	1	
Table 9
OP\IP	00	01	10	11	
00	0	0	1	1	
01	0	0	0	0	
10	1	1	0	0	
11	0	0	0	0	
Table 10
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	1	1	1	
11	0	0	0	0	
Table 11
OP\IP	00	01	10	11	
00	0	0	1	0	
01	0	0	0	1	
10	1	0	0	0	
11	0	1	0	0	
Table 12
OP\IP	00	01	10	11	
00	0	0	0	0	
01	0	0	0	0	
10	1	0	1	0	
11	0	1	0	1	
Table 13
OP\IP	00	01	10	11	
00	0	0	0	1	
01	0	0	1	0	
10	0	1	0	0	


In [11]:
print(AllUniqueORs)
print(len(AllUniqueORs))

[[[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 1, 1], [0, 0, 0, 0]], [[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0]], [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], [[0, 1, 0, 1], [1, 0, 1, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], [[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], [[0, 0, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 1, 1]], [[0, 0, 1, 1], [0, 0, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]], [[0, 0, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1]], [[0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0]], [[0, 0, 0, 0], [0, 0, 0, 0], [0, 1, 0, 1], [1, 0, 1, 0]], [[0, 0, 0, 0], [0, 0, 1, 1], [0, 0, 0, 0], [1, 1, 0, 0]], [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 1, 1], [0, 0, 0, 0]], [[1, 1, 1, 1]

In [12]:
NonClassicalPNS=[]
NonClassicalPNSCode=[]
for table in range(len(AllPNSTables)):
    if AllPNSTables[table] not in AllUniqueORs:
        NonClassicalPNS.append(AllPNSTables[table])
        NonClassicalPNSCode.append(AllPNSCode[table])
print("The number of non-classical PNS tabes is:", len(NonClassicalPNS))

The number of non-classical PNS tabes is: 1240
